# 07_v5c_dynamic_tqqq — V5C 3.1 + 动态 TQQQ 估值择时（方案 B）

> **方案 B**：使用 NDX 历史 P/E TTM 真分位（不是价格区间位置）

## 策略机制

| NDX P/E 在 5Y 的真分位 | TQQQ 权重 | QQQ 权重 |
|---|---|---|
| 0-20% (便宜) | 7.5% | 7.5% |
| 20-50% (偏低) | 5.0% | 10% |
| 50-80% (偏高) | 2.5% | 12.5% |
| 80-100% (昂贵) | 0% | 15% |

## 关键改进 vs 方案 A

1. **真分位**（rank-based），不是 min-max 区间位置——单日极值不再扭曲
2. **基于 P/E**（估值），不是绝对价格——剔除 EPS 增长造成的虚假信号

## 数据局限

- NDX 月度 P/E 历史由公开数据源整理，**月度精度**，估值约 ±2x 误差
- 长期趋势和大幅波动方向可信，对回测策略概念**够用**
- 如果策略验证有效，可替换为更精确的日度 P/E 数据

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from io import StringIO

# ============================================================
# NDX 月度 P/E TTM 历史 (2011-01 至 2026-05, 公开数据源整理)
# 来源参考: multpl.com / Macrotrends / 财报会议披露
# 精度: 月度近似, 约 ±2x 误差范围
# ============================================================
ndx_pe_csv = """date,pe
2011-01-01,17.5
2011-04-01,18.0
2011-07-01,17.0
2011-10-01,15.5
2012-01-01,15.0
2012-04-01,16.5
2012-07-01,16.0
2012-10-01,15.5
2013-01-01,15.5
2013-04-01,17.0
2013-07-01,18.0
2013-10-01,19.0
2014-01-01,19.5
2014-04-01,20.5
2014-07-01,21.0
2014-10-01,20.5
2015-01-01,21.5
2015-04-01,22.0
2015-07-01,22.5
2015-10-01,21.5
2016-01-01,21.0
2016-04-01,21.5
2016-07-01,22.5
2016-10-01,22.0
2017-01-01,22.5
2017-04-01,24.0
2017-07-01,25.0
2017-10-01,26.0
2018-01-01,26.5
2018-04-01,25.0
2018-07-01,26.0
2018-10-01,24.0
2018-12-01,21.0
2019-01-01,21.5
2019-04-01,24.0
2019-07-01,25.0
2019-10-01,25.5
2020-01-01,28.0
2020-03-01,22.0
2020-04-01,24.0
2020-07-01,32.0
2020-10-01,34.0
2020-12-01,37.0
2021-01-01,37.0
2021-04-01,35.0
2021-07-01,32.0
2021-10-01,30.0
2021-12-01,30.0
2022-01-01,29.0
2022-04-01,26.0
2022-07-01,24.0
2022-10-01,22.0
2022-12-01,22.5
2023-01-01,23.0
2023-04-01,26.0
2023-07-01,28.0
2023-10-01,27.0
2023-12-01,29.0
2024-01-01,29.5
2024-04-01,30.5
2024-07-01,31.0
2024-10-01,30.5
2024-12-01,32.0
2025-01-01,32.5
2025-04-01,30.0
2025-07-01,31.5
2025-10-01,32.5
2025-12-01,33.0
2026-01-01,33.5
2026-03-01,33.0
2026-04-01,33.0
2026-05-01,33.0
"""

ndx_pe_monthly = pd.read_csv(StringIO(ndx_pe_csv), parse_dates=['date'], index_col='date')['pe']
print(f'NDX P/E 数据点数: {len(ndx_pe_monthly)}')
print(f'范围: {ndx_pe_monthly.index[0].date()} -> {ndx_pe_monthly.index[-1].date()}')
print(f'\n统计:')
print(f'  最小: {ndx_pe_monthly.min():.1f}x  (日期: {ndx_pe_monthly.idxmin().date()})')
print(f'  最大: {ndx_pe_monthly.max():.1f}x  (日期: {ndx_pe_monthly.idxmax().date()})')
print(f'  中位数: {ndx_pe_monthly.median():.1f}x')
print(f'  当前: {ndx_pe_monthly.iloc[-1]:.1f}x')

In [ ]:
# ============================================================
# 拉取价格数据 + 把 P/E 插值到日度
# ============================================================
tickers_long = {
    'VOO': 'VFINX', 'QQQ': 'QQQ', 'TQQQ': 'TQQQ',
    'HQH': 'HQH', 'XLV': 'XLV', 'BCX': 'BCX',
    'GLDM': 'GLD', 'VGSH': 'VFITX',
}

start_date = '2011-01-01'
raw = yf.download(list(tickers_long.values()), start=start_date, auto_adjust=True)['Close']
rename = {v: k for k, v in tickers_long.items()}
raw.columns = [rename.get(c, c) for c in raw.columns]
data = raw.dropna()
returns = data.pct_change().dropna()

# 把月度 P/E 重采样到日度（线性插值）
ndx_pe_daily = ndx_pe_monthly.reindex(
    pd.date_range(start=data.index[0], end=data.index[-1], freq='D')
).interpolate(method='linear').ffill().bfill()
ndx_pe_daily = ndx_pe_daily.reindex(data.index, method='ffill')

print(f'日度 P/E 序列: {len(ndx_pe_daily)} 天')
print(f'\n关键时点 P/E:')
for d in ['2012-01-01', '2018-12-31', '2020-03-23', '2021-12-31', '2022-10-12', '2026-05-01']:
    if pd.Timestamp(d) in ndx_pe_daily.index:
        print(f'  {d}: {ndx_pe_daily.loc[d]:.1f}x')
    else:
        nearest = ndx_pe_daily.index.get_indexer([pd.Timestamp(d)], method='nearest')[0]
        print(f'  {d} (nearest {ndx_pe_daily.index[nearest].date()}): {ndx_pe_daily.iloc[nearest]:.1f}x')

In [ ]:
# ============================================================
# 计算 P/E 5Y 真分位 (rank-based, 非 min-max)
# ============================================================
lookback = 252 * 5  # 5 年滚动窗口

# rank(pct=True) 给出真正的分位值（每个点在窗口内的排名占比）
pe_percentile = ndx_pe_daily.rolling(lookback).rank(pct=True)

# 起始 5 年没有完整数据, 用窗口内已有数据计算（min_periods=252 = 1 年起算）
pe_percentile_partial = ndx_pe_daily.rolling(lookback, min_periods=252).rank(pct=True)
pe_percentile = pe_percentile.fillna(pe_percentile_partial).fillna(0.5)

# 权重映射
def tqqq_weight(p):
    if p < 0.20:   return 0.075
    elif p < 0.50: return 0.05
    elif p < 0.80: return 0.025
    else:          return 0.0

tqqq_weights_series = pe_percentile.apply(tqqq_weight)
qqq_weights_series = 0.15 - tqqq_weights_series

# 可视化
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

axes[0].plot(ndx_pe_daily.index, ndx_pe_daily.values, linewidth=1.2, color='C0')
axes[0].set_title('NDX P/E TTM 历史', fontsize=12)
axes[0].set_ylabel('P/E')
axes[0].grid(alpha=0.3)
axes[0].axhline(ndx_pe_daily.median(), color='gray', linestyle='--', alpha=0.5, label=f'中位数 {ndx_pe_daily.median():.1f}x')
axes[0].legend()

axes[1].plot(pe_percentile.index, pe_percentile.values, linewidth=1.2, color='C1')
axes[1].axhline(0.20, color='green', linestyle='--', alpha=0.5, label='便宜区 (20%)')
axes[1].axhline(0.80, color='red', linestyle='--', alpha=0.5, label='昂贵区 (80%)')
axes[1].set_title('NDX P/E 5Y 滚动真分位 (rank-based)', fontsize=12)
axes[1].set_ylabel('Percentile')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].fill_between(tqqq_weights_series.index, tqqq_weights_series.values, color='red', alpha=0.5)
axes[2].set_title('动态 TQQQ 权重 (基于 P/E 真分位)', fontsize=12)
axes[2].set_ylabel('Weight')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 当前状态
current_pe = ndx_pe_daily.iloc[-1]
current_pct = pe_percentile.iloc[-1]
current_tqqq = tqqq_weights_series.iloc[-1]
print(f'\n当前状态 ({ndx_pe_daily.index[-1].date()}):')
print(f'  NDX P/E: {current_pe:.1f}x')
print(f'  5Y 真分位: {current_pct:.1%}')
print(f'  TQQQ 目标权重: {current_tqqq:.1%}')

print(f'\n各分位区间历史出现频率 (5Y 完整窗口期):')
valid_pct = pe_percentile.iloc[lookback:]
for label, lo, hi in [('便宜 (<20%)', 0, 0.2), ('偏低 (20-50%)', 0.2, 0.5),
                       ('偏高 (50-80%)', 0.5, 0.8), ('昂贵 (>80%)', 0.8, 1.01)]:
    pct = ((valid_pct >= lo) & (valid_pct < hi)).mean()
    print(f'  {label:<18}: {pct:.1%}')

In [ ]:
# ============================================================
# simulate_rebalance 支持时变目标
# ============================================================
def simulate_rebalance_dynamic(returns_df, base_weights, dynamic_overrides=None, threshold_pp=5.0):
    used = list(base_weights.keys())
    if dynamic_overrides:
        for t in dynamic_overrides:
            if t not in used: used.append(t)
    used = [t for t in used if t in returns_df.columns]
    sub_returns = returns_df[used]
    
    target = np.array([base_weights.get(t, 0) for t in used])
    if dynamic_overrides:
        for t, series in dynamic_overrides.items():
            idx = used.index(t)
            target[idx] = series.iloc[0]
    target = target / target.sum()
    
    current_weights = target.copy()
    portfolio_returns = []
    rebalance_dates = [sub_returns.index[0]]
    
    for date, daily_ret in sub_returns.iterrows():
        if dynamic_overrides:
            cur_target = np.array([base_weights.get(t, 0) for t in used])
            for t, series in dynamic_overrides.items():
                idx = used.index(t)
                cur_target[idx] = series.loc[date] if date in series.index else cur_target[idx]
            cur_target = cur_target / cur_target.sum()
        else:
            cur_target = target
        
        port_ret = np.sum(current_weights * daily_ret.values)
        portfolio_returns.append(port_ret)
        new_weights = current_weights * (1 + daily_ret.values)
        new_weights = new_weights / new_weights.sum()
        
        max_dev_pp = np.max(np.abs(new_weights - cur_target)) * 100
        if max_dev_pp >= threshold_pp:
            current_weights = cur_target.copy()
            rebalance_dates.append(date)
        else:
            current_weights = new_weights
    
    return pd.Series(portfolio_returns, index=sub_returns.index), rebalance_dates


def compute_metrics(returns_series, rebalance_dates, name='Portfolio'):
    cum = (1 + returns_series).cumprod()
    n_years = len(returns_series) / 252
    cagr = cum.iloc[-1] ** (1/n_years) - 1
    vol = returns_series.std() * np.sqrt(252)
    sharpe = (cagr - 0.04) / vol
    downside = returns_series[returns_series < 0]
    sortino = (cagr - 0.04) / (downside.std() * np.sqrt(252))
    rolling_max = cum.expanding().max()
    drawdown = (cum / rolling_max) - 1
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd)
    return {
        'Name': name, 'CAGR': cagr, 'Vol': vol,
        'Sharpe': sharpe, 'Sortino': sortino,
        'Max DD': max_dd, 'Calmar': calmar,
        'Rebalances': len(rebalance_dates) - 1,
    }

In [ ]:
# ============================================================
# 三组对比
# ============================================================

# A: V5C 3.1 baseline
V5C_3_1 = {
    'VOO': 0.15, 'QQQ': 0.15, 'HQH': 0.10, 'XLV': 0.10,
    'GLDM': 0.20, 'BCX': 0.10, 'VGSH': 0.20,
}

# B: 静态 TQQQ 5%
V5C_3_1_STATIC = {
    'VOO': 0.15, 'QQQ': 0.10, 'TQQQ': 0.05,
    'HQH': 0.10, 'XLV': 0.10,
    'GLDM': 0.20, 'BCX': 0.10, 'VGSH': 0.20,
}

# C: 动态 TQQQ (基于 P/E 真分位)
V5C_3_1_DYNAMIC_BASE = {
    'VOO': 0.15, 'HQH': 0.10, 'XLV': 0.10,
    'GLDM': 0.20, 'BCX': 0.10, 'VGSH': 0.20,
}
DYNAMIC_OVERRIDES = {
    'QQQ': qqq_weights_series,
    'TQQQ': tqqq_weights_series,
}

ret_A, dates_A = simulate_rebalance_dynamic(returns, V5C_3_1, threshold_pp=5.0)
ret_B, dates_B = simulate_rebalance_dynamic(returns, V5C_3_1_STATIC, threshold_pp=5.0)
ret_C, dates_C = simulate_rebalance_dynamic(returns, V5C_3_1_DYNAMIC_BASE,
                                              dynamic_overrides=DYNAMIC_OVERRIDES,
                                              threshold_pp=5.0)

mA = compute_metrics(ret_A, dates_A, 'V5C 3.1 (基准)')
mB = compute_metrics(ret_B, dates_B, 'V5C 3.1 + 静态 TQQQ')
mC = compute_metrics(ret_C, dates_C, 'V5C 3.1 + 动态 TQQQ (P/E)')

df = pd.DataFrame([mA, mB, mC]).set_index('Name')
print('='*82)
print('三组对比 (±5pp 阈值再平衡, 14.6 年回测)')
print('='*82)
for col in ['CAGR', 'Vol', 'Sharpe', 'Sortino', 'Max DD', 'Calmar']:
    fmt = '{:.2%}' if col not in ['Sharpe','Sortino','Calmar'] else '{:.3f}'
    a = df.loc['V5C 3.1 (基准)', col]
    b = df.loc['V5C 3.1 + 静态 TQQQ', col]
    c = df.loc['V5C 3.1 + 动态 TQQQ (P/E)', col]
    print(f'  {col:<10}  A: {fmt.format(a):>10}  B: {fmt.format(b):>10}  C: {fmt.format(c):>10}')
print(f'\n  Rebalances: A: {mA["Rebalances"]}  B: {mB["Rebalances"]}  C: {mC["Rebalances"]}')

In [ ]:
# ============================================================
# 净值与回撤
# ============================================================
cum_A = (1 + ret_A).cumprod()
cum_B = (1 + ret_B).cumprod()
cum_C = (1 + ret_C).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
axes[0].plot(cum_A, label='A: V5C 3.1 基准', linewidth=2, alpha=0.85)
axes[0].plot(cum_B, label='B: V5C 3.1 + 静态 TQQQ 5%', linewidth=2, alpha=0.85)
axes[0].plot(cum_C, label='C: V5C 3.1 + 动态 TQQQ (P/E)', linewidth=2, alpha=0.85)
axes[0].set_title('净值曲线 (log scale)', fontsize=14)
axes[0].set_yscale('log')
axes[0].legend()
axes[0].grid(alpha=0.3)

for cum, label, color in [(cum_A, 'A', 'C0'), (cum_B, 'B', 'C1'), (cum_C, 'C', 'C2')]:
    rm = cum.expanding().max()
    dd = (cum / rm) - 1
    axes[1].fill_between(dd.index, dd.values, 0, alpha=0.3, color=color, label=label)
axes[1].set_title('回撤对比', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 危机期 + 黄金区表现
# ============================================================
events = {
    '2018-Q4 跌势':        ('2018-10-01', '2018-12-31'),
    '2019-2020 上涨':      ('2019-01-01', '2020-02-19'),
    '2020 COVID':          ('2020-02-19', '2020-04-30'),
    '2020-2021 反弹':      ('2020-04-30', '2021-12-31'),
    '2022 Bear (1-10月)':  ('2022-01-01', '2022-10-31'),
    '2023-2024 复苏':      ('2023-01-01', '2024-12-31'),
    '2025-2026 高位':      ('2025-01-01', '2026-04-30'),
}

print('='*78)
print('各时期表现对比')
print('='*78)
print(f'{"时期":<22} {"A":>10} {"B":>10} {"C":>10} {"C-A":>10}')
print('-'*78)
for name, (s, e) in events.items():
    a = (1 + ret_A.loc[s:e]).prod() - 1
    b = (1 + ret_B.loc[s:e]).prod() - 1
    c = (1 + ret_C.loc[s:e]).prod() - 1
    delta_ca = c - a
    print(f'{name:<22} {a:>+9.2%}  {b:>+9.2%}  {c:>+9.2%}  {delta_ca:>+9.2%}')

In [ ]:
# ============================================================
# 动态权重激活分析: TQQQ 在每个区间的累计加权天数
# ============================================================
regime_days = pd.cut(pe_percentile, bins=[0, 0.2, 0.5, 0.8, 1.01],
                       labels=['便宜<20%', '偏低 20-50%', '偏高 50-80%', '昂贵>80%'])
regime_counts = regime_days.value_counts().sort_index()
regime_periods = regime_counts.apply(lambda x: f'{x} 天 ({x/252:.1f} 年)')

print('='*60)
print('14.6 年内 NDX P/E 处于各区间的天数')
print('='*60)
for region, days in regime_periods.items():
    pct = regime_counts[region] / regime_counts.sum()
    weight = {'便宜<20%': 0.075, '偏低 20-50%': 0.05, '偏高 50-80%': 0.025, '昂贵>80%': 0.0}[region]
    print(f'  {region:<14} {days:<20}  ({pct:.1%}) → TQQQ {weight:.1%}')

## 解读模板

**核心问题：动态择时是否创造价值？**

看几对比较：

1. **C vs A**：动态 TQQQ 是否优于纯指数？
   - C 的 Sharpe ≥ A 的 Sharpe → ✅ 择时有正收益
   - C 的 CAGR > A 但 Sharpe < A → ⚠️ 收益靠加杠杆，不是择时

2. **C vs B**：动态择时是否优于静态加杠杆？
   - C 显著优于 B → ✅ 择时本身有价值（不只是杠杆）
   - C ≈ B → ❌ 择时没用，杠杆是收益源

3. **2022 Bear 期间 C 的表现**：
   - 这是策略最该发挥的窗口（P/E 下行 → TQQQ 应增仓）
   - 如果 C 在 2022 没明显跑赢 A/B，策略机制有问题

4. **当前状态（2026-05）**：
   - 当前 P/E 33x 在 5Y 高分位 → TQQQ 应该 0%
   - 落地实际权重 = V5C 3.1（这个策略今天激活率为 0）

## 决策标准

**采纳 V5C 3.2（动态 TQQQ）** 仅当：
- C Sharpe > A Sharpe **且** C Max DD 不显著恶化
- C 在 2022/2018-Q4 等下行窗口至少不输 A
- 当前 P/E 进入便宜区时（2-5 年内可能发生）能自动激活杠杆